# AutoShot embeddings per video

Version này dùng output từ `get-keyframe-autoshot.ipynb` và lưu theo format:

- `features/<vit-model>/<video_id>.npy`
- `map-keyframes/<video_id>.csv`

Mỗi file `.npy` chứa embedding của toàn bộ keyframes trong một folder/video. File CSV tương ứng giữ metadata tối giản: `n`, `pts_time`, `fps`, `frame_idx`.

In [ ]:
!pip install -q open_clip_torch

In [ ]:
from pathlib import Path
import json
import time
import zipfile
from datetime import datetime

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import open_clip
from tqdm.auto import tqdm

## Config

`DATASET_ROOT` là Kaggle dataset bạn đã tạo từ AutoShot output. Nếu dataset đang nằm trực tiếp dạng folder thì notebook dùng luôn. Nếu chỉ có `autoshot_output.zip` thì notebook unzip sang `/kaggle/working/autoshot_input`.

In [ ]:
DATASET_ROOT = Path('/kaggle/input/datasets/khngxuninh/autoshot-output')
ARCHIVE_PATH = DATASET_ROOT / 'autoshot_output.zip'
AUTOSHOT_ROOT = Path('/kaggle/working/autoshot_input')

OUTPUT_ROOT = Path('/kaggle/working/embedding_per_video')

# Chiến lược nhanh, ổn cho baseline retrieval text-image bằng OpenCLIP.
MODEL_NAME = 'ViT-B-32'
PRETRAINED = 'laion2b_s34b_b79k'
BATCH_SIZE = 256
NUM_WORKERS = 2

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

MODEL_FOLDER = 'vit-' + MODEL_NAME.replace('/', '-') + '-' + PRETRAINED.replace('/', '-')
FEATURE_DIR = OUTPUT_ROOT / 'features' / MODEL_FOLDER
MAP_DIR = OUTPUT_ROOT / 'map-keyframes'

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)

print('DEVICE:', DEVICE)
print('FEATURE_DIR:', FEATURE_DIR)
print('MAP_DIR:', MAP_DIR)

## Load AutoShot output

In [ ]:
if not (AUTOSHOT_ROOT / 'shot_segments.csv').exists():
    if ARCHIVE_PATH.exists():
        AUTOSHOT_ROOT.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(ARCHIVE_PATH, 'r') as zf:
            zf.extractall(AUTOSHOT_ROOT)
    else:
        AUTOSHOT_ROOT = DATASET_ROOT

csv_candidates = list(AUTOSHOT_ROOT.rglob('shot_segments.csv'))
csv_path = csv_candidates[0]
AUTOSHOT_ROOT = csv_path.parent
df = pd.read_csv(csv_path)

if 'saved' in df.columns:
    df = df[df['saved'].astype(str).str.lower().isin(['true', '1', 'yes'])].copy()

df['video_id'] = df['image_path'].apply(lambda x: Path(str(x)).parent.name)
df['filename'] = df['image_path'].apply(lambda x: Path(str(x)).name)
df['local_image_path'] = df.apply(lambda r: AUTOSHOT_ROOT / 'frames' / r['video_id'] / r['filename'], axis=1)

sort_cols = ['video_id', 'shot_id', 'frame_idx', 'frame_type']
df = df.sort_values([c for c in sort_cols if c in df.columns]).reset_index(drop=True)

print('AutoShot root:', AUTOSHOT_ROOT)
print('Total keyframes:', len(df))
print('Total videos:', df['video_id'].nunique())
display(df.head())

## Dataset + model

In [ ]:
class ImagePathDataset(Dataset):
    def __init__(self, paths, preprocess):
        self.paths = list(paths)
        self.preprocess = preprocess

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = Image.open(self.paths[idx]).convert('RGB')
        return self.preprocess(image)


model, _, preprocess = open_clip.create_model_and_transforms(
    MODEL_NAME,
    pretrained=PRETRAINED,
    device=DEVICE,
)
model.eval()

print('Loaded:', MODEL_NAME, PRETRAINED)

## Encode và lưu từng video

Output CSV có đúng các cột:

- `n`: thứ tự keyframe trong video, bắt đầu từ 1
- `pts_time`: lấy từ `frame_sec`
- `fps`: lấy từ cột `fps` nếu AutoShot có lưu
- `frame_idx`: index frame gốc trong video

Vector thứ `n-1` trong `.npy` tương ứng với dòng `n` trong CSV.

In [ ]:
start_all = time.time()
video_infos = []

for video_id, g in tqdm(df.groupby('video_id', sort=True), desc='Videos'):
    g = g.reset_index(drop=True)
    image_paths = g['local_image_path'].tolist()

    loader = DataLoader(
        ImagePathDataset(image_paths, preprocess),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=(DEVICE == 'cuda'),
    )

    batches = []
    t0 = time.time()
    with torch.no_grad():
        for images in loader:
            images = images.to(DEVICE, non_blocking=True)
            emb = model.encode_image(images, normalize=True)
            batches.append(emb.cpu().numpy().astype('float32'))

    embeddings = np.concatenate(batches, axis=0)

    feature_path = FEATURE_DIR / f'{video_id}.npy'
    map_path = MAP_DIR / f'{video_id}.csv'

    np.save(feature_path, embeddings)

    map_df = pd.DataFrame({
        'n': np.arange(1, len(g) + 1, dtype=np.int64),
        'pts_time': g['frame_sec'].astype(float),
        'fps': g['fps'].astype(float) if 'fps' in g.columns else np.nan,
        'frame_idx': g['frame_idx'].astype(np.int64),
    })
    map_df.to_csv(map_path, index=False)

    video_infos.append({
        'video_id': video_id,
        'num_keyframes': len(g),
        'embedding_shape': list(embeddings.shape),
        'feature_path': str(feature_path),
        'map_path': str(map_path),
        'seconds': round(time.time() - t0, 3),
    })

summary = pd.DataFrame(video_infos)
summary_path = OUTPUT_ROOT / 'per_video_summary.csv'
summary.to_csv(summary_path, index=False)

print('Done in minutes:', round((time.time() - start_all) / 60, 2))
print('Summary:', summary_path)
display(summary.head())

## Save model info

In [ ]:
model_info = {
    'created_at': datetime.utcnow().isoformat() + 'Z',
    'dataset_root': str(DATASET_ROOT),
    'autoshot_root': str(AUTOSHOT_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'feature_dir': str(FEATURE_DIR),
    'map_dir': str(MAP_DIR),
    'model_name': MODEL_NAME,
    'pretrained': PRETRAINED,
    'model_folder': MODEL_FOLDER,
    'device': DEVICE,
    'batch_size': BATCH_SIZE,
    'num_workers': NUM_WORKERS,
    'l2_normalized': True,
    'dtype': 'float32',
    'num_videos': int(df['video_id'].nunique()),
    'num_keyframes': int(len(df)),
    'csv_columns': ['n', 'pts_time', 'fps', 'frame_idx'],
    'npy_layout': 'one file per video; row index = csv n - 1',
}

info_path = OUTPUT_ROOT / 'model_info.json'
info_path.write_text(json.dumps(model_info, indent=2, ensure_ascii=False), encoding='utf-8')

print(info_path)
model_info

## Kiểm tra nhanh một video

In [ ]:
sample_video = summary.iloc[0]['video_id']
sample_features = np.load(FEATURE_DIR / f'{sample_video}.npy')
sample_map = pd.read_csv(MAP_DIR / f'{sample_video}.csv')

print(sample_video, sample_features.shape)
display(sample_map.head())